# Feature Engineering: Creating Better Features from Data

> "Coming up with features is difficult, time-consuming, requires expert knowledge. Applied machine learning is basically feature engineering." — Andrew Ng

This notebook covers the art and science of feature engineering — transforming raw data into features that better represent the underlying problem to the predictive models.

---
## 1. What is Feature Engineering and Why It Matters More Than Model Choice

**Feature engineering** is the process of transforming raw data into inputs (features) that machine learning algorithms can understand and learn from effectively.

### Why it matters:
- Better features → simpler models → better generalization
- Domain knowledge injected through features beats blind tuning
- A linear model with great features often beats a deep network with raw features (on small-to-medium data)
- Kaggle competitions are won on features, not model architectures

In [ ]:
# Core imports used throughout the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (
    PolynomialFeatures, LabelEncoder, StandardScaler, MinMaxScaler
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, r2_score
from sklearn.inspection import permutation_importance

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

---
## 2. Numerical Feature Engineering

### 2.1 Polynomial Features (Interaction & Squared Terms)

Capture non-linear relationships between features by creating $x_1^2$, $x_1 x_2$, etc.

In [ ]:
# Synthetic data: y = 3 + 2*x1 - 0.5*x2 + 0.8*x1*x2 + 1.5*x1^2 + noise
np.random.seed(42)
n = 500
X1 = np.random.uniform(-2, 2, n)
X2 = np.random.uniform(-2, 2, n)
y = (3 + 2*X1 - 0.5*X2 + 0.8*X1*X2 + 1.5*X1**2 + np.random.normal(0, 0.5, n))

df_poly = pd.DataFrame({'x1': X1, 'x2': X2, 'y': y})

# Compare linear vs polynomial
X_raw = df_poly[['x1', 'x2']].values
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_raw)
feature_names = poly.get_feature_names_out(['x1', 'x2'])

print("Original features:", X_raw.shape[1])
print("Polynomial features:", X_poly.shape[1])
print("Feature names:", list(feature_names))

lin_reg = LinearRegression()
lin_reg.fit(X_raw, y)
y_pred_lin = lin_reg.predict(X_raw)

poly_reg = LinearRegression()
poly_reg.fit(X_poly, y)
y_pred_poly = poly_reg.predict(X_poly)

print(f"\nLinear R²: {r2_score(y, y_pred_lin):.4f}")
print(f"Polynomial R²: {r2_score(y, y_pred_poly):.4f}")

In [ ]:
# Visualize the fit improvement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y, y_pred_lin, alpha=0.5)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
axes[0].set_title(f'Linear Model (R² = {r2_score(y, y_pred_lin):.3f})')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')

axes[1].scatter(y, y_pred_poly, alpha=0.5, color='green')
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
axes[1].set_title(f'Polynomial Model (R² = {r2_score(y, y_pred_poly):.3f})')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')

plt.tight_layout()
plt.show()

### 2.2 Binning / Discretization

Convert continuous variables into categorical bins. Common strategies: equal-width, equal-frequency (quantile), and custom bins.

In [ ]:
# Demonstrate binning strategies
np.random.seed(42)
age_data = np.random.exponential(scale=35, size=1000)
age_data = np.clip(age_data, 0, 100).astype(int)

# Equal-width bins
bins_equal_width = pd.cut(age_data, bins=5, labels=['0-20', '20-40', '40-60', '60-80', '80-100'])

# Equal-frequency (quantile) bins
bins_equal_freq = pd.qcut(age_data, q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])

# Custom bins (domain-driven)
custom_bins = pd.cut(age_data, bins=[0, 12, 18, 35, 50, 65, 100],
                     labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Middle-Aged', 'Senior'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bins_equal_width.value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Equal-Width Bins')
axes[0].set_ylabel('Count')

bins_equal_freq.value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Equal-Frequency (Quantile) Bins')

custom_bins.value_counts().sort_index().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Domain-Specific Custom Bins')

plt.tight_layout()
plt.show()

print("Equal-width counts:")
print(bins_equal_width.value_counts().sort_index())
print("\nEqual-frequency counts (should be ~200 each):")
print(bins_equal_freq.value_counts().sort_index())

### 2.3 Log, Exponential, and Power Transformations

Handle skewed distributions, stabilize variance, and linearize relationships.

In [ ]:
# Log transformation for right-skewed data
np.random.seed(42)
income = np.random.lognormal(mean=10, sigma=0.8, size=1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(income, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Original Income Distribution (Right-Skewed)')
axes[0].set_xlabel('Income')

axes[1].hist(np.log1p(income), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log-Transformed (log1p)')
axes[1].set_xlabel('log(Income)')

plt.tight_layout()
plt.show()

from scipy.stats import skew
print(f"Skewness original: {skew(income):.3f}")
print(f"Skewness log-transformed: {skew(np.log1p(income)):.3f}")

### 2.4 Ratio Features

Ratios often capture relationships between variables better than raw values (e.g., debt-to-income, BMI).

In [ ]:
# Ratio features example
np.random.seed(42)
df_ratio = pd.DataFrame({
    'income': np.random.lognormal(10, 0.5, 500),
    'debt': np.random.lognormal(8, 0.7, 500),
    'assets': np.random.lognormal(11, 0.6, 500),
    'expenses': np.random.lognormal(9, 0.4, 500)
})

# Create ratio features
df_ratio['debt_to_income'] = df_ratio['debt'] / df_ratio['income']
df_ratio['savings_ratio'] = (df_ratio['income'] - df_ratio['expenses']) / df_ratio['income']
df_ratio['asset_to_debt'] = df_ratio['assets'] / (df_ratio['debt'] + 1)

print("Ratio feature statistics:")
print(df_ratio[['debt_to_income', 'savings_ratio', 'asset_to_debt']].describe())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, col in enumerate(['debt_to_income', 'savings_ratio', 'asset_to_debt']):
    axes[i].hist(df_ratio[col].clip(0, df_ratio[col].quantile(0.95)), bins=30, edgecolor='white')
    axes[i].set_title(col.replace('_', ' ').title())
plt.tight_layout()
plt.show()

### 2.5 Aggregation Features (Group Statistics)

Aggregate information across groups: mean, std, min, max, count per category.

In [ ]:
# Simulate transactional data
np.random.seed(42)
customers = [f'CUST_{i}' for i in range(1, 101)]
categories = ['Electronics', 'Clothing', 'Food', 'Books', 'Sports']

df_trans = pd.DataFrame({
    'customer_id': np.random.choice(customers, size=2000),
    'category': np.random.choice(categories, size=2000),
    'amount': np.random.exponential(scale=50, size=2000),
    'quantity': np.random.randint(1, 10, size=2000)
})

# Aggregate features per customer
customer_agg = df_trans.groupby('customer_id').agg({
    'amount': ['mean', 'std', 'sum', 'count'],
    'quantity': 'mean'
})
customer_agg.columns = ['_'.join(col).strip() for col in customer_agg.columns.values]
customer_agg = customer_agg.reset_index()

print("Customer-level aggregation features:")
print(customer_agg.head())
print(f"\nShape: {customer_agg.shape}")

---
## 3. Categorical Feature Engineering

### 3.1 Frequency Encoding

Replace categories with their frequency in the training set. Useful for high-cardinality features.

In [ ]:
# Frequency encoding
np.random.seed(42)
cities = ['New York', 'London', 'Tokyo', 'Paris', 'Berlin', 'Sydney']
probs = [0.35, 0.25, 0.15, 0.12, 0.08, 0.05]

df_cat = pd.DataFrame({'city': np.random.choice(cities, size=2000, p=probs)})

freq_map = df_cat['city'].value_counts().to_dict()
df_cat['city_freq'] = df_cat['city'].map(freq_map)

print("Frequency encoding map:")
for city, freq in sorted(freq_map.items(), key=lambda x: -x[1]):
    print(f"  {city:15s} → {freq:5d} ({freq/len(df_cat):.1%})")
print(f"\nFirst 10 rows:\n{df_cat.head(10)}")

### 3.2 Target Encoding (with Smoothing)

Replace category with the mean of the target for that category. Smoothing prevents overfitting for rare categories.

In [ ]:
# Target encoding with smoothing
np.random.seed(42)
n = 2000
categories = ['A', 'B', 'C', 'D', 'E']
cat_probs = [0.40, 0.25, 0.18, 0.12, 0.05]

df_te = pd.DataFrame({
    'category': np.random.choice(categories, size=n, p=cat_probs),
    'target': np.random.binomial(1, 0.3, n)
})
# Make target depend somewhat on category
for i, cat in enumerate(categories):
    mask = df_te['category'] == cat
    base_prob = 0.2 + 0.05 * i
    df_te.loc[mask, 'target'] = np.random.binomial(1, base_prob, mask.sum())

# Target encoding with smoothing
global_mean = df_te['target'].mean()
m = 30  # smoothing factor (higher = more shrinkage to global mean)

cat_stats = df_te.groupby('category')['target'].agg(['mean', 'count'])
cat_stats['smoothed'] = (
    (cat_stats['mean'] * cat_stats['count'] + global_mean * m) /
    (cat_stats['count'] + m)
)

print("Target encoding with smoothing (m=30):")
print("  Cat  Raw Mean  Count  Smoothed")
for cat in categories:
    row = cat_stats.loc[cat]
    print(f"  {cat:5s}  {row['mean']:.4f}  {int(row['count']):5d}  {row['smoothed']:.4f}")

df_te['target_encoded'] = df_te['category'].map(cat_stats['smoothed'])
print(f"\nGlobal mean: {global_mean:.4f}")
print(f"Smoothing pulls rare categories toward global mean")

### 3.3 Ordinal Encoding (Custom Mappings)

For ordinal categories with a natural order (e.g., education level, rating scale).

In [ ]:
# Ordinal encoding with custom mapping
education_order = {
    'PhD': 5,
    'Master': 4,
    'Bachelor': 3,
    'Associate': 2,
    'High School': 1,
    'None': 0
}

np.random.seed(42)
levels = list(education_order.keys())
probs_edu = [0.05, 0.10, 0.30, 0.15, 0.30, 0.10]

df_ord = pd.DataFrame({
    'education': np.random.choice(levels, size=500, p=probs_edu)
})
df_ord['education_ordinal'] = df_ord['education'].map(education_order)

print("Ordinal encoding of education:")
print(df_ord.head(10))
print(f"\nMapping: {education_order}")

### 3.4 Count Encoding

Replace category with the count of occurrences. Similar to frequency encoding but the raw count is used.

In [ ]:
# Count encoding
np.random.seed(42)
countries = ['USA', 'Canada', 'UK', 'Australia', 'India', 'Germany', 'France', 'Japan']
probs_country = [0.25, 0.10, 0.15, 0.05, 0.20, 0.10, 0.08, 0.07]

df_cnt = pd.DataFrame({'country': np.random.choice(countries, size=1000, p=probs_country)})
count_map = df_cnt['country'].value_counts().to_dict()
df_cnt['country_count'] = df_cnt['country'].map(count_map)

df_cnt['country_label'] = df_cnt['country'].astype('category').cat.codes

print("Count encoding:")
print(df_cnt.groupby('country').first().reset_index())
print(f"\nSample:\n{df_cnt.head(10)}")

---
## 4. Date / Time Feature Engineering

### 4.1 Extracting Temporal Components

In [ ]:
# Date feature extraction
np.random.seed(42)
dates = pd.date_range('2022-01-01', '2024-12-31', freq='D')
random_dates = np.random.choice(dates, size=500)

df_dates = pd.DataFrame({'timestamp': random_dates})
df_dates['year'] = df_dates['timestamp'].dt.year
df_dates['month'] = df_dates['timestamp'].dt.month
df_dates['day'] = df_dates['timestamp'].dt.day
df_dates['dayofweek'] = df_dates['timestamp'].dt.dayofweek  # 0=Monday
df_dates['quarter'] = df_dates['timestamp'].dt.quarter
df_dates['is_weekend'] = df_dates['dayofweek'].isin([5, 6]).astype(int)
df_dates['dayofyear'] = df_dates['timestamp'].dt.dayofyear
df_dates['hour'] = np.random.randint(0, 24, size=500)

df_dates['weekday_name'] = df_dates['timestamp'].dt.day_name()

print("Date/time component extraction:")
print(df_dates.head(10))

### 4.2 Cyclical Encoding (Sin/Cos Transform)

Preserve the cyclical nature of time features (hours, months, weekdays) using sine and cosine transformations.

In [ ]:
# Cyclical encoding
hours = np.arange(0, 24)
months = np.arange(1, 13)

df_cycle = pd.DataFrame({'hour': hours, 'month': months})

df_cycle['hour_sin'] = np.sin(2 * np.pi * df_cycle['hour'] / 24)
df_cycle['hour_cos'] = np.cos(2 * np.pi * df_cycle['hour'] / 24)
df_cycle['month_sin'] = np.sin(2 * np.pi * (df_cycle['month'] - 1) / 12)
df_cycle['month_cos'] = np.cos(2 * np.pi * (df_cycle['month'] - 1) / 12)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(df_cycle['hour'], df_cycle['hour_sin'], 'o-', label='sin(hour)', color='coral')
axes[0].plot(df_cycle['hour'], df_cycle['hour_cos'], 's-', label='cos(hour)', color='steelblue')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Encoded Value')
axes[0].set_title('Cyclical Encoding of Hour (24-hour cycle)')
axes[0].legend()
axes[0].set_xticks([0, 6, 12, 18, 23])
axes[0].grid(True, alpha=0.3)

circle = plt.Circle((0, 0), 1, fill=False, linestyle='--', alpha=0.3, color='gray')
ax = axes[1]
sc = ax.scatter(df_cycle['hour_sin'], df_cycle['hour_cos'], c=df_cycle['hour'],
                cmap='viridis', s=80, edgecolor='k', linewidth=0.5)
ax.add_patch(circle)
ax.set_xlabel('sin(hour)')
ax.set_ylabel('cos(hour)')
ax.set_title('Hour on Unit Circle (preserves circular continuity)')
ax.set_aspect('equal')
cbar = plt.colorbar(sc, ax=ax, label='Hour')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: 23:00 and 0:00 are close on the unit circle, preserving their natural similarity.")

### 4.3 Time Since Event & Rolling Window Features

In [ ]:
# Time since event and rolling features
np.random.seed(42)
date_range = pd.date_range('2023-01-01', periods=365, freq='D')
values = np.random.randn(365).cumsum() + 100

df_ts = pd.DataFrame({'date': date_range, 'value': values})

# Time since a specific event
event_date = pd.Timestamp('2023-06-15')
df_ts['days_since_event'] = (df_ts['date'] - event_date).dt.days

# Rolling window features
df_ts['rolling_mean_7'] = df_ts['value'].rolling(7).mean()
df_ts['rolling_std_7'] = df_ts['value'].rolling(7).std()
df_ts['rolling_max_14'] = df_ts['value'].rolling(14).max()
df_ts['rolling_min_14'] = df_ts['value'].rolling(14).min()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_ts['date'], df_ts['value'], alpha=0.5, label='Original', linewidth=1)
ax.plot(df_ts['date'], df_ts['rolling_mean_7'], label='7-day Rolling Mean', linewidth=2)
ax.fill_between(df_ts['date'],
                df_ts['rolling_mean_7'] - df_ts['rolling_std_7'],
                df_ts['rolling_mean_7'] + df_ts['rolling_std_7'],
                alpha=0.2, label='±1 Std Dev')
ax.axvline(event_date, color='red', linestyle='--', label='Event Date')
ax.set_title('Rolling Window Features and Time Since Event')
ax.set_xlabel('Date')
ax.set_ylabel('Value')
ax.legend()
plt.show()

---
## 5. Text Feature Engineering

### 5.1 Basic Text Statistics

In [ ]:
# Basic text features
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is transforming every industry.",
    "Feature engineering is the key to success in data science.",
    "Python is a versatile programming language for data analysis.",
    "AI"
]

df_text = pd.DataFrame({'text': texts})
df_text['char_count'] = df_text['text'].str.len()
df_text['word_count'] = df_text['text'].str.split().str.len()
df_text['avg_word_length'] = df_text['char_count'] / df_text['word_count']
df_text['capital_ratio'] = (
    df_text['text'].str.findall(r'[A-Z]').str.len() / df_text['char_count']
)
df_text['num_sentences'] = df_text['text'].str.count(r'\.') + df_text['text'].str.count(r'\!') + df_text['text'].str.count(r'\?')

print("Basic text features:")
print(df_text)

### 5.2 TF-IDF Features

Term Frequency — Inverse Document Frequency captures how important a word is to a document in a corpus.

In [ ]:
# TF-IDF features
corpus = [
    "machine learning deep learning neural networks",
    "feature engineering data preprocessing cleaning",
    "machine learning model evaluation validation testing",
    "deep neural networks are powerful models",
    "data cleaning and feature engineering are crucial"
]

tfidf = TfidfVectorizer(max_features=10, stop_words='english')
X_tfidf = tfidf.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

print("TF-IDF feature matrix (limited to top 10 terms):")
print(df_tfidf)

print(f"\nVocabulary: {list(tfidf.get_feature_names_out())}")

### 5.3 N-Grams

Capture short word sequences (n-grams) to preserve context.

In [ ]:
# N-gram features
texts_ng = [
    "not good",
    "very good",
    "not bad",
    "very bad",
    "good product"
]

from sklearn.feature_extraction.text import CountVectorizer

# Bigrams
bigram_vec = CountVectorizer(ngram_range=(2, 2), stop_words='english')
X_bigram = bigram_vec.fit_transform(texts_ng)

df_bigram = pd.DataFrame(
    X_bigram.toarray(),
    columns=bigram_vec.get_feature_names_out()
)
df_bigram['text'] = texts_ng

print("Bigram features:")
print(df_bigram)

# Combined unigrams + bigrams
combined_vec = CountVectorizer(ngram_range=(1, 2), stop_words='english')
X_combined = combined_vec.fit_transform(texts_ng)
print(f"\nCombined unigram+bigram shape: {X_combined.shape}")
print(f"Features: {list(combined_vec.get_feature_names_out())}")

---
## 6. Interaction Features

Product, sum, or difference of two features can capture relationships that individual features miss.

In [ ]:
# Manual interaction features
np.random.seed(42)
df_inter = pd.DataFrame({
    'height_cm': np.random.normal(170, 10, 300),
    'weight_kg': np.random.normal(70, 15, 300),
    'age_years': np.random.randint(18, 70, 300),
    'hours_exercise': np.random.exponential(3, 300)
})

df_inter['bmi'] = df_inter['weight_kg'] / ((df_inter['height_cm'] / 100) ** 2)
df_inter['height_weight_product'] = df_inter['height_cm'] * df_inter['weight_kg']
df_inter['exercise_age_interaction'] = df_inter['hours_exercise'] * df_inter['age_years']
df_inter['height_diff_from_mean'] = df_inter['height_cm'] - df_inter['height_cm'].mean()

corr_matrix = df_inter.corr()['bmi'].sort_values(ascending=False)
print("Correlation with BMI (a derived interaction feature):")
print(corr_matrix)

# PolynomialFeatures can also create interactions automatically
poly_int = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_int = poly_int.fit_transform(df_inter[['height_cm', 'weight_kg']])
print(f"\nInteraction features from height & weight: {poly_int.get_feature_names_out(['height', 'weight'])}")

---
## 7. Domain-Specific Features

> "Domain knowledge is the best feature engineer"

### Examples across domains:

| Domain | Raw Data | Engineered Feature |
|--------|----------|-------------------|
| Finance | Stock price | Moving avg crossover, RSI, volatility |
| Healthcare | Patient age, symptoms | Comorbidity index, risk score |
| E-commerce | Purchase history | Recency-Frequency-Monetary (RFM) score |
| NLP | Review text | Sentiment polarity, subjectivity |
| Geospatial | Lat, Long | Distance to nearest hospital, cluster ID |
| Cybersecurity | Network logs | Connection frequency, unusual port score |

**The best features come from understanding the problem deeply.**

Spend time with domain experts — they know which relationships matter.

---
## 8. Automated Feature Engineering with Featuretools (Conceptual)

[Featuretools](https://www.featuretools.com/) uses **Deep Feature Synthesis (DFS)** to automatically generate features from relational data.

### Key Concepts:
- **Entities**: Tables in your data
- **Relationships**: How tables connect (foreign keys)
- **Primitives**: Transformation (e.g., `month`, `weekday`) and aggregation (e.g., `mean`, `count`) operations
- **DFS**: Automatically stacks primitives to create deep features

```python
# Conceptual example (requires featuretools installed)
import featuretools as ft

es = ft.EntitySet(id='customers')
es = es.add_dataframe(...)
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='customers',
    max_depth=2
)
```

> **When to use**: You have relational data (customers → transactions → items) and want to quickly generate hundreds of candidate features.

---
## 9. Feature Scaling as Part of Engineering

**When to scale:**
- **Always scale** for: SVM, k-NN, PCA, Neural Networks, Gradient Descent-based models
- **Not needed for**: Tree-based models (Random Forest, XGBoost, LightGBM)
- **For regression**: Scale when using regularization (Ridge, Lasso) or when comparing coefficient magnitudes

**Common scalers:**
- `StandardScaler`: (x - mean) / std — for normally distributed data
- `MinMaxScaler`: (x - min) / (max - min) — for bounded data
- `RobustScaler`: (x - median) / IQR — for data with outliers
- `MaxAbsScaler`: x / |max| — for sparse data

In [ ]:
# Demonstrate scaling impact
np.random.seed(42)
df_scale = pd.DataFrame({
    'age': np.random.randint(18, 80, 200),
    'income': np.random.lognormal(10, 0.6, 200),
    'score': np.random.uniform(0, 100, 200)
})

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_scale),
    columns=df_scale.columns
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

df_scale.boxplot(ax=ax1)
ax1.set_title('Original Features (different scales)')
ax1.set_ylabel('Value')

df_scaled.boxplot(ax=ax2)
ax2.set_title('After StandardScaler (mean=0, std=1)')
ax2.set_ylabel('Standardized Value')

plt.tight_layout()
plt.show()

print("Before scaling:")
print(df_scale.describe().loc[['mean', 'std']])
print("\nAfter scaling:")
print(df_scaled.describe().loc[['mean', 'std']])

---
## 10. Validation: Does the Engineered Feature Actually Help?

Engineered features are not free — each one adds complexity and risk of overfitting. Always validate.

In [ ]:
# Validate feature engineering with model comparison and feature importance
np.random.seed(42)
n = 1000

# Base features
X_base = pd.DataFrame({
    'feature_a': np.random.randn(n),
    'feature_b': np.random.randn(n),
    'feature_c': np.random.randn(n)
})

# Target depends on base + interaction + squared term
y_val = (
    0.5 * X_base['feature_a'] +
    0.3 * X_base['feature_b'] +
    0.8 * X_base['feature_a'] * X_base['feature_b'] +
    0.6 * X_base['feature_c'] ** 2 +
    np.random.normal(0, 0.3, n)
)

# Add engineered features
X_eng = X_base.copy()
X_eng['a_b_interaction'] = X_eng['feature_a'] * X_eng['feature_b']
X_eng['c_squared'] = X_eng['feature_c'] ** 2
X_eng['a_plus_b'] = X_eng['feature_a'] + X_eng['feature_b']
X_eng['b_minus_c'] = X_eng['feature_b'] - X_eng['feature_c']

X_train_b, X_test_b, X_train_e, X_test_e, y_train, y_test = train_test_split(
    X_base, X_eng, y_val, test_size=0.3, random_state=42
)

rf_base = RandomForestRegressor(n_estimators=100, random_state=42)
rf_eng = RandomForestRegressor(n_estimators=100, random_state=42)

rf_base.fit(X_train_b, y_train)
rf_eng.fit(X_train_e, y_train)

y_pred_base = rf_base.predict(X_test_b)
y_pred_eng = rf_eng.predict(X_test_e)

print("Model comparison — Base vs. Engineered features:")
print(f"  Base features R²:       {r2_score(y_test, y_pred_base):.4f}")
print(f"  Engineered features R²: {r2_score(y_test, y_pred_eng):.4f}")
print(f"  Improvement:            {r2_score(y_test, y_pred_eng) - r2_score(y_test, y_pred_base):.4f}")

# Feature importance
importances = pd.DataFrame({
    'feature': X_eng.columns,
    'importance': rf_eng.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(importances['feature'], importances['importance'], color='steelblue')
ax.set_title('Feature Importance with Engineered Features')
ax.set_xlabel('Importance')
for i, v in enumerate(importances['importance']):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.show()

print("\nEngineered features (interaction, squared) show high importance, validating their addition.")

---
## 11. Avoiding Data Leakage in Feature Engineering

**Data leakage** happens when information from outside the training set (especially from the target or test set) is used to create features, leading to over-optimistic performance.

### Common leakage sources in feature engineering:

| Leaky Practice | Why It's Leaky | Safe Alternative |
|----------------|----------------|------------------|
| Target encoding before train/test split | Test set target influences training | Compute encoding on training fold only |
| Scaling before splitting | Test statistics influence training | Fit scaler on training set, transform both |
| Rolling features using future data | Future values leak into past | Use expanding window, never look ahead |
| Imputation using full dataset statistics | Test data affects imputation values | Compute statistics on training set only |
| Feature selection on full data | Target relationship overestimated | Select features inside CV loop |
| Group aggregation on full data | Target of test rows in same group affects stats | Aggregate per fold or use leave-one-group-out |

```python
# Safe approach: compute everything inside train set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Compute encoding on TRAIN only, then map to TEST
target_mean = y_train.groupby(X_train['category']).mean()
X_train['target_encoded'] = X_train['category'].map(target_mean)
X_test['target_encoded']  = X_test['category'].map(target_mean).fillna(global_mean)

# Fit scaler on TRAIN only, transform both
scaler = StandardScaler().fit(X_train[['num_col']])
X_train['num_col_scaled'] = scaler.transform(X_train[['num_col']])
X_test['num_col_scaled']  = scaler.transform(X_test[['num_col']])
```

---
## 12. Complete Example: Titanic Dataset

Feature engineering from scratch on a classic dataset.

In [ ]:
# Load Titanic data
import os

titanic_path = r'D:\4-1\AI\git_ml\data-science-mastery\07-advanced-techniques\titanic.csv'
if os.path.exists(titanic_path):
    df = pd.read_csv(titanic_path)
else:
    # Load from seaborn if available
    try:
        df = sns.load_dataset('titanic')
        # Rename to match titanic convention
        df = df.rename(columns={'sex': 'Sex', 'age': 'Age', 'sibsp': 'SibSp',
                                'parch': 'Parch', 'fare': 'Fare', 'embarked': 'Embarked',
                                'class': 'Pclass', 'who': 'Who', 'adult_male': 'AdultMale',
                                'deck': 'Deck', 'embark_town': 'EmbarkTown',
                                'alive': 'Alive', 'alone': 'Alone'})
        df['Survived'] = df['survived'].astype(int)
        df['Pclass'] = df['pclass'].astype(int)
        df['Name'] = ''
    except:
        print("Downloading Titanic dataset...")
        url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
        df = pd.read_csv(url)

print(f"Titanic dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

In [ ]:
# Feature engineering on Titanic
df_eng = df.copy()

# 1. Title extraction from Name
df_eng['Title'] = df_eng['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major',
               'Rev', 'Sir', 'Jonkheer', 'Dona', 'Mme', 'Ms', 'Mlle']
df_eng['Title'] = df_eng['Title'].replace(rare_titles, 'Rare')
df_eng['Title'] = df_eng['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
title_map = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Rare': 5}
df_eng['Title_ordinal'] = df_eng['Title'].map(title_map).fillna(0)

# 2. Family size
df_eng['FamilySize'] = df_eng['SibSp'] + df_eng['Parch'] + 1
df_eng['IsAlone'] = (df_eng['FamilySize'] == 1).astype(int)

# 3. Cabin letter (deck)
df_eng['CabinLetter'] = df_eng['Cabin'].str[0].fillna('U')

# 4. Age group
df_eng['AgeGroup'] = pd.cut(df_eng['Age'], bins=[0, 12, 18, 35, 50, 65, 100],
                             labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'MiddleAge', 'Senior'])

# 5. Fare binning
df_eng['FareBin'] = pd.qcut(df_eng['Fare'].fillna(df_eng['Fare'].median()),
                             q=4, labels=['Low', 'Medium', 'High', 'VeryHigh'], duplicates='drop')

# 6. Is male (simple encoding)
df_eng['IsMale'] = (df_eng['Sex'] == 'male').astype(int) if 'Sex' in df_eng.columns else 0

# 7. Embarked fill
df_eng['Embarked'] = df_eng['Embarked'].fillna('S')
embarked_map = {'S': 0, 'C': 1, 'Q': 2}
df_eng['Embarked_encoded'] = df_eng['Embarked'].map(embarked_map).fillna(0)

# Show engineered features
eng_cols = ['Title', 'Title_ordinal', 'FamilySize', 'IsAlone',
            'CabinLetter', 'AgeGroup', 'FareBin', 'IsMale', 'Embarked_encoded']
print("Engineered features (sample):")
print(df_eng[eng_cols].head(10))
print(f"\nNumber of new features created: {len(eng_cols)}")

In [ ]:
# Correlation before vs after feature engineering
# Build correlation datasets
numeric_base = ['Age', 'Fare', 'SibSp', 'Parch']
if 'Survived' in df_eng.columns:
    numeric_base = ['Survived'] + numeric_base

available_base = [c for c in numeric_base if c in df_eng.columns]
corr_base = df_eng[available_base].corr()

# Engineered numeric features
eng_numeric = ['Title_ordinal', 'FamilySize', 'IsAlone', 'IsMale', 'Embarked_encoded', 'Age', 'Fare']
if 'Survived' in df_eng.columns:
    eng_numeric = ['Survived'] + eng_numeric
available_eng = [c for c in eng_numeric if c in df_eng.columns]
corr_eng = df_eng[available_eng].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(corr_base, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=axes[0], cbar=False)
axes[0].set_title('Correlation — Base Features Only')

sns.heatmap(corr_eng, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=axes[1])
axes[1].set_title('Correlation — With Engineered Features')

plt.tight_layout()
plt.show()

print("Engineered features (Title_ordinal, IsAlone, FamilySize) show stronger correlation with survival.")

In [ ]:
# Model performance: with vs without feature engineering on Titanic
if 'Survived' in df_eng.columns:
    # Prepare base features
    base_features = ['Age', 'Fare']
    base_features = [c for c in base_features if c in df_eng.columns]

    # Prepare engineered features (including base where needed)
    eng_feature_cols = ['Title_ordinal', 'FamilySize', 'IsAlone',
                        'IsMale', 'Embarked_encoded', 'Age', 'Fare']
    eng_feature_cols = [c for c in eng_feature_cols if c in df_eng.columns]

    # Drop rows with NaN
    df_model = df_eng[['Survived'] + eng_feature_cols].dropna()

    X_base_m = df_model[base_features]
    X_eng_m = df_model[eng_feature_cols]
    y_m = df_model['Survived']

    X_train_b, X_test_b, X_train_e, X_test_e, y_train_m, y_test_m = train_test_split(
        X_base_m, X_eng_m, y_m, test_size=0.3, random_state=42
    )

    # Impute missing (shouldn't be any after dropna, but safe)
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy='median')
    X_train_b = imp.fit_transform(X_train_b)
    X_test_b = imp.transform(X_test_b)
    imp_e = SimpleImputer(strategy='median')
    X_train_e = imp_e.fit_transform(X_train_e)
    X_test_e = imp_e.transform(X_test_e)

    rf_base_m = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_eng_m = RandomForestClassifier(n_estimators=100, random_state=42)

    rf_base_m.fit(X_train_b, y_train_m)
    rf_eng_m.fit(X_train_e, y_train_m)

    acc_base = accuracy_score(y_test_m, rf_base_m.predict(X_test_b))
    acc_eng = accuracy_score(y_test_m, rf_eng_m.predict(X_test_e))

    cv_base = cross_val_score(rf_base_m, X_train_b, y_train_m, cv=5).mean()
    cv_eng = cross_val_score(rf_eng_m, X_train_e, y_train_m, cv=5).mean()

    results_df = pd.DataFrame({
        'Metric': ['Test Accuracy', 'CV Score (mean)'],
        'Base Features': [f'{acc_base:.3f}', f'{cv_base:.3f}'],
        'With Engineered Features': [f'{acc_eng:.3f}', f'{cv_eng:.3f}'],
        'Improvement': [f'{acc_eng - acc_base:+.3f}', f'{cv_eng - cv_base:+.3f}']
    })
    print("Model Performance Comparison — Titanic")
    print(results_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 5))
    metrics = ['Test Accuracy', 'CV Score (mean)']
    base_scores = [acc_base, cv_base]
    eng_scores = [acc_eng, cv_eng]

    x = np.arange(len(metrics))
    width = 0.35
    bars1 = ax.bar(x - width/2, base_scores, width, label='Base Features', color='steelblue')
    bars2 = ax.bar(x + width/2, eng_scores, width, label='With Engineered Features', color='coral')

    ax.set_ylabel('Score')
    ax.set_title('Model Performance: Base vs. Engineered Features')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.set_ylim(0.5, 1.0)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom')
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()
else:
    print("Survived column not found; skipping model comparison.")

---
## Summary: Feature Engineering Checklist

- [ ] Did I extract all relevant **numerical transformations** (log, polynomial, ratios)?
- [ ] Did I encode **categorical** variables appropriately (not just one-hot)?
- [ ] Did I extract meaningful **date/time** features?
- [ ] Did I consider **interaction** features?
- [ ] Did I use **domain knowledge** to create custom features?
- [ ] Did I **validate** that each feature helps (feature importance, CV)?
- [ ] Did I avoid **data leakage**?
- [ ] Did I **scale** appropriately for the model?

---
## Practice Exercises

1. **Ratio Features**: Load the Boston Housing dataset (or any real estate dataset). Create features like `rooms_per_household`, `bedroom_ratio`, `population_density`. Compare model performance before and after adding these features.

2. **Target Encoding**: Take any classification dataset with a high-cardinality categorical feature (e.g., zip code, user ID). Implement target encoding with different smoothing factors (m=5, 10, 50, 100). Show how smoothing affects overfitting using cross-validation.

3. **Cyclical + Lag Features**: Use a time series dataset (e.g., hourly energy consumption, bike sharing). Engineer cyclical features for hour and day-of-week, plus lag features (t-1, t-2, t-24). Train a model with and without these features and compare performance.

4. **Titanic Challenge**: Starting from the raw Titanic dataset, engineer at least 5 features not shown in this notebook (e.g., ticket prefix grouping, name length, deck grouping, fare per person, group survival rate). Build a classifier and achieve the best CV score you can. Log which features help and which don't.

5. **Interaction Search**: Use `PolynomialFeatures(interaction_only=True)` on the Iris dataset. Compute the mutual information gain of each interaction feature. Which interactions are most informative? Visualize the top-3 interaction features in pairplots.